# Moneta — Datenbank-Schicht (`db.py`)

Diese Datei ist die **unterste Schicht** der App. Alle anderen Teile gehen über diese Datei,
wenn sie Daten lesen oder speichern wollen.

**Was diese Datei macht:**
- Verbindung zur SQLite-Datenbank aufbauen (optional verschlüsselt)
- Datenbankschema definieren (welche Tabellen es gibt)
- Standard-Kategorien und Regeln beim ersten Start anlegen

> ⚠️ Diese Datei definiert nur Funktionen — einzelne Zellen können ausgeführt werden,
> aber `init_db()` braucht eine laufende Datenbankdatei unter `~/.moneta/data.db`.

## Zelle 1 · Imports

Wir laden nur Bibliotheken aus der Python-Standardbibliothek — keine externen Abhängigkeiten hier.

- **`uuid`** — erzeugt weltweit eindeutige IDs (z.B. `a3f1c2d4-1234-...`) als Primärschlüssel
- **`datetime`** — für Zeitstempel (`created_at`, `updated_at`)
- **`pathlib.Path`** — plattformunabhängige Dateipfade, z.B. `Path.home() / ".moneta" / "data.db"`
- **`sqlite3`** — Pythons eingebaute SQLite-Bibliothek, wird als Fallback genutzt

In [ ]:
import uuid
from datetime import datetime
from pathlib import Path
import sqlite3 as _plain_sqlite3

## Zelle 2 · Verschlüsselungsschicht: SQLCipher oder plain SQLite

Moneta unterstützt **optionale Datenbankverschlüsselung** mit SQLCipher (AES-256).

Das `try/except` ist ein *bedingter Import*:
- **SQLCipher verfügbar** → `_db_module = sqlcipher3`, `ENCRYPTION_AVAILABLE = True`
- **SQLCipher fehlt** → Fallback auf normale `sqlite3`, keine Verschlüsselung

Dieses Muster heißt **graceful degradation**: Die App funktioniert in beiden Fällen,
ist aber sicherer wenn SQLCipher installiert ist.

Der Rest des Codes benutzt immer `_db_module.connect(...)` — egal welche Bibliothek dahinter steckt.

In [ ]:
try:
    from sqlcipher3 import dbapi2 as _db_module
    ENCRYPTION_AVAILABLE = True
except ImportError:
    import sqlite3 as _db_module  # type: ignore
    ENCRYPTION_AVAILABLE = False

## Zelle 3 · Globale Konfiguration: Datenbankpfad und Passwort

- **`DB_PATH`** — wo die Datenbankdatei liegt.
  `Path.home()` gibt das Home-Verzeichnis zurück (z.B. `/Users/clarabrilke`).
  Die Datenbank liegt also unter `~/.moneta/data.db` — **außerhalb** des Projektordners,
  damit sie nicht aus Versehen mit dem Code gelöscht wird.

- **`DB_KEY`** — das Passwort im RAM. Startet als `None` (kein Passwort gesetzt).
  Wird **nie** auf die Festplatte geschrieben. Lebt nur so lange die App läuft.

In [ ]:
DB_PATH = Path.home() / ".moneta" / "data.db"
DB_KEY: str | None = None

## Zelle 4 · Passwort merken (`set_db_key`)

Diese Funktion speichert das Passwort in der globalen Variable `DB_KEY`.

`global DB_KEY` ist nötig, weil wir innerhalb einer Funktion eine *globale* Variable
ändern wollen — ohne dieses Schlüsselwort würde Python eine neue lokale Variable anlegen.

Das Passwort lebt nur im RAM: beim Beenden der App ist es weg.

In [ ]:
def set_db_key(key: str) -> None:
    """Store the passphrase in memory. Never written to disk."""
    global DB_KEY
    DB_KEY = key

## Zelle 5 · Datenbankverbindung öffnen (`get_db`)

Diese Funktion öffnet jedes Mal eine **neue Verbindung** zur Datenbankdatei.
Sie wird bei jedem API-Aufruf aufgerufen und danach wieder geschlossen (`with get_db() as conn:`).

**Schritt für Schritt:**
1. `mkdir(parents=True, exist_ok=True)` — legt `~/.moneta/` an, falls noch nicht vorhanden
2. `connect(str(DB_PATH))` — öffnet (oder erstellt) die Datenbankdatei
3. `PRAGMA key='...'` — **muss die erste Operation sein** bei SQLCipher; entschlüsselt die Datei
4. `row_factory = Row` — damit können Spalten per Name angesprochen werden: `row['name']` statt `row[0]`
5. `PRAGMA journal_mode=WAL` — Write-Ahead-Logging: bessere Performance bei gleichzeitigen Zugriffen
6. `PRAGMA foreign_keys=ON` — erzwingt referentielle Integrität (z.B. Transaktion braucht ein existierendes Konto)

In [ ]:
def get_db():
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    conn = _db_module.connect(str(DB_PATH))
    if ENCRYPTION_AVAILABLE and DB_KEY:
        escaped = DB_KEY.replace("'", "''")
        conn.execute(f"PRAGMA key='{escaped}'")
    conn.row_factory = _db_module.Row
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("PRAGMA foreign_keys=ON")
    return conn

## Zelle 6 · Bestehende Datenbank verschlüsseln (`migrate_encrypt_if_needed`)

Wer Moneta vor der Verschlüsselungsfunktion genutzt hat, hat eine **unverschlüsselte** `data.db`.
Diese Funktion erkennt das und verschlüsselt sie **automatisch** beim ersten Start mit Passwort.

**Erkennungslogik:**
- Normales `sqlite3` kann die Datei öffnen → sie ist unverschlüsselt
- `sqlite3` scheitert → sie ist bereits verschlüsselt (oder kaputt) → nichts tun

**Migrationsprozess:**
1. SQLCipher öffnet die unverschlüsselte Datei **ohne** Passwort (Kompatibilitätsmodus)
2. `PRAGMA rekey='passwort'` verschlüsselt die gesamte Datei **in-place**
3. Ab jetzt braucht jede Verbindung das Passwort

Es wird **kein zweites Exemplar** angelegt — die Originaldatei wird direkt überschrieben.

In [ ]:
def migrate_encrypt_if_needed(key: str) -> bool:
    if not ENCRYPTION_AVAILABLE or not DB_PATH.exists():
        return False

    try:
        test = _plain_sqlite3.connect(str(DB_PATH))
        test.execute("SELECT 1 FROM sqlite_master")
        test.close()
    except _plain_sqlite3.DatabaseError:
        return False  # already encrypted, or corrupt

    conn = _db_module.connect(str(DB_PATH))
    escaped = key.replace("'", "''")
    conn.execute(f"PRAGMA rekey='{escaped}'")
    conn.close()
    return True

## Zelle 7 · Passwort prüfen (`verify_key`)

Prüft beim App-Start, ob das eingegebene Passwort zur Datenbank passt.

Wenn das Passwort falsch ist, scheitert `SELECT 1 FROM sqlite_master` — SQLCipher
kann die verschlüsselte Datei nicht lesen und wirft eine Exception.

Diese Funktion wird einmalig beim Start aufgerufen. Bei falschem Passwort beendet
sich die App sofort (bevor der Server startet und Daten preisgibt).

In [ ]:
def verify_key() -> bool:
    if not DB_PATH.exists():
        return True
    try:
        conn = get_db()
        conn.execute("SELECT 1 FROM sqlite_master")
        conn.close()
        return True
    except Exception:
        return False

## Zelle 8 · Hilfsfunktionen: IDs und Zeitstempel

- **`uid()`** — erzeugt eine zufällige UUID v4, z.B. `'3f7a1c2d-4e5b-6789-abcd-...'`.
  Wird als Primärschlüssel für jeden Datensatz verwendet.
  Vorteil gegenüber fortlaufenden Zahlen (1, 2, 3...): keine Konflikte beim Import/Export,
  keine Rate-my-ID-Angriffe.

- **`now_iso()`** — aktueller Zeitstempel im ISO-8601-Format, z.B. `'2026-06-26T14:23:01.123456'`.
  Wird für `created_at` und `updated_at` verwendet.

In [ ]:
def uid() -> str:
    return str(uuid.uuid4())

def now_iso() -> str:
    return datetime.now().isoformat()

## Zelle 9 · Datenbankschema — Tabellendefinitionen

Das Schema definiert alle Tabellen. `CREATE TABLE IF NOT EXISTS` bedeutet:
Die Tabelle wird nur angelegt, wenn sie noch nicht existiert —
so kann dieses SQL beim Start immer ausgeführt werden, ohne Daten zu löschen.

**Tabellen:**
| Tabelle | Zweck |
|---------|-------|
| `accounts` | Konten (Girokonto, Sparkonto, Bargeld...) |
| `categories` | Kategorien mit Über-/Untergruppen-Hierarchie (self-referencing) |
| `transactions` | Alle Buchungen (Einnahmen, Ausgaben, Umbuchungen) |
| `recurring_rules` | Wiederkehrende Buchungen (monatlich, jährlich...) |
| `category_rules` | Auto-Kategorisierungsregeln (z.B. 'REWE' → Supermarkt) |
| `budgets` | Budgetgrenzen pro Kategorie |
| `savings_goals` | Sparziele mit Zielbetrag und Datum |
| `pots` | Virtuelle Töpfe innerhalb eines Kontos |
| `settings` | Allgemeine App-Einstellungen als Key-Value-Paare |

`ON DELETE CASCADE` bei `transactions.account_id` bedeutet:
Wenn ein Konto gelöscht wird, werden automatisch alle zugehörigen Transaktionen mitgelöscht.

In [ ]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS accounts (
    id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    type TEXT NOT NULL,
    initial_balance REAL NOT NULL DEFAULT 0.0,
    currency TEXT NOT NULL DEFAULT 'EUR',
    created_at TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS categories (
    id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    parent_id TEXT REFERENCES categories(id) ON DELETE SET NULL,
    type TEXT NOT NULL,
    icon TEXT DEFAULT '📁',
    created_at TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS transactions (
    id TEXT PRIMARY KEY,
    account_id TEXT NOT NULL REFERENCES accounts(id) ON DELETE CASCADE,
    amount REAL NOT NULL,
    date TEXT NOT NULL,
    payee TEXT DEFAULT '',
    purpose TEXT DEFAULT '',
    note TEXT DEFAULT '',
    category_id TEXT REFERENCES categories(id) ON DELETE SET NULL,
    type TEXT NOT NULL,
    transfer_to_account_id TEXT REFERENCES accounts(id),
    recurring_rule_id TEXT,
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS recurring_rules (
    id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    amount REAL NOT NULL,
    type TEXT NOT NULL,
    category_id TEXT REFERENCES categories(id) ON DELETE SET NULL,
    account_id TEXT NOT NULL REFERENCES accounts(id) ON DELETE CASCADE,
    payee TEXT DEFAULT '',
    purpose TEXT DEFAULT '',
    interval_type TEXT NOT NULL,
    day_of_month INTEGER DEFAULT 1,
    next_due_date TEXT NOT NULL,
    active INTEGER NOT NULL DEFAULT 1,
    created_at TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS category_rules (
    id TEXT PRIMARY KEY,
    pattern TEXT NOT NULL,
    field TEXT NOT NULL,
    category_id TEXT NOT NULL REFERENCES categories(id) ON DELETE CASCADE,
    priority INTEGER DEFAULT 0,
    created_at TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS budgets (
    id TEXT PRIMARY KEY,
    category_id TEXT NOT NULL UNIQUE REFERENCES categories(id) ON DELETE CASCADE,
    amount REAL NOT NULL,
    created_at TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS savings_goals (
    id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    target_amount REAL NOT NULL,
    current_amount REAL NOT NULL DEFAULT 0.0,
    monthly_contribution REAL NOT NULL DEFAULT 0.0,
    target_date TEXT,
    color TEXT DEFAULT '#6366f1',
    created_at TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS pots (
    id TEXT PRIMARY KEY,
    account_id TEXT NOT NULL REFERENCES accounts(id) ON DELETE CASCADE,
    name TEXT NOT NULL,
    target_amount REAL NOT NULL DEFAULT 0.0,
    color TEXT DEFAULT '#6366f1',
    created_at TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS settings (
    key TEXT PRIMARY KEY,
    value TEXT NOT NULL
);
"""

## Zelle 10 · Standard-Kategorien

Beim allerersten Start (leere Datenbank) werden diese Kategorien automatisch angelegt.
So hat jeder Nutzer sofort eine sinnvolle Grundstruktur — ohne selbst etwas anlegen zu müssen.

**Format:** `(interner_schlüssel, Anzeigename, Elternschlüssel, Typ, Icon)`

- `Elternschlüssel = None` → Übergruppe (z.B. "Fixkosten", "Lebensmittel")
- `Elternschlüssel = 'fixkosten'` → Untergruppe (z.B. "Miete" unter "Fixkosten")
- `type = 'expense'` oder `'income'` → Ausgabe oder Einnahme

Der interne Schlüssel wird nur während des Seeds verwendet, um Eltern-Kind-Beziehungen
herzustellen. In der Datenbank werden UUIDs als echte IDs gespeichert.

In [ ]:
DEFAULT_CATEGORIES = [
    # (slug_key, name, parent_slug, type, icon)
    # Übergruppen – Ausgaben
    ("fixkosten",        "Fixkosten",               None,           "expense", "🏠"),
    ("abos",             "Abos & Mitgliedschaften",  None,           "expense", "🔄"),
    ("lebensmittel",     "Lebensmittel",             None,           "expense", "🛒"),
    ("lifestyle",        "Lifestyle & Freizeit",     None,           "expense", "🎉"),
    ("transport",        "Transport",                None,           "expense", "🚗"),
    ("gesundheit",       "Gesundheit",               None,           "expense", "💊"),
    ("sonstiges_a",      "Sonstiges",                None,           "expense", "📦"),
    # Übergruppen – Einnahmen
    ("gehalt_cat",       "Gehalt & Lohn",            None,           "income",  "💰"),
    ("einnahmen_s",      "Sonstige Einnahmen",       None,           "income",  "💵"),
    # Untergruppen – Ausgaben
    ("miete",            "Miete / Hypothek",         "fixkosten",    "expense", "🏠"),
    ("strom",            "Strom",                    "fixkosten",    "expense", "⚡"),
    ("gas",              "Gas / Heizung",            "fixkosten",    "expense", "🔥"),
    ("internet",         "Internet",                 "fixkosten",    "expense", "🌐"),
    ("telefon",          "Telefon",                  "fixkosten",    "expense", "📱"),
    ("rundfunk",         "Rundfunkbeitrag",          "fixkosten",    "expense", "📺"),
    ("versicherung",     "Versicherungen",           "fixkosten",    "expense", "🛡️"),
    ("streaming",        "Streaming",                "abos",         "expense", "🎬"),
    ("zeitungen",        "Zeitungen / Magazine",     "abos",         "expense", "📰"),
    ("software",         "Software / Cloud",         "abos",         "expense", "💻"),
    ("fitness",          "Fitnessstudio",            "abos",         "expense", "🏋️"),
    ("musik",            "Musik-Abo",                "abos",         "expense", "🎵"),
    ("supermarkt",       "Supermarkt",               "lebensmittel", "expense", "🛒"),
    ("baeckerei",        "Bäckerei / Markt",         "lebensmittel", "expense", "🥐"),
    ("restaurant",       "Restaurant & Café",        "lifestyle",    "expense", "🍽️"),
    ("bar",              "Bar & Ausgehen",           "lifestyle",    "expense", "🍻"),
    ("kino",             "Kino & Theater",           "lifestyle",    "expense", "🎭"),
    ("shopping",         "Shopping & Kleidung",      "lifestyle",    "expense", "👗"),
    ("hobby",            "Hobby",                    "lifestyle",    "expense", "🎨"),
    ("reisen",           "Urlaub & Reisen",          "lifestyle",    "expense", "✈️"),
    ("opnv",             "ÖPNV / Bahn",              "transport",    "expense", "🚌"),
    ("tanken",           "Tanken",                   "transport",    "expense", "⛽"),
    ("taxi",             "Taxi / Ridesharing",       "transport",    "expense", "🚕"),
    ("arzt",             "Arzt & Apotheke",          "gesundheit",   "expense", "👨‍⚕️"),
    # Untergruppen – Einnahmen
    ("gehalt",           "Gehalt",                   "gehalt_cat",   "income",  "💼"),
    ("freelance",        "Freelance",                "gehalt_cat",   "income",  "💡"),
    ("stipendium",       "Stipendium",               "gehalt_cat",   "income",  "🎓"),
    ("zinsen",           "Zinsen & Dividenden",      "einnahmen_s",  "income",  "📈"),
    ("steuer",           "Steuerrückerstattung",     "einnahmen_s",  "income",  "🏛️"),
    ("geschenke",        "Geschenke erhalten",       "einnahmen_s",  "income",  "🎁"),
]

## Zelle 11 · Standard-Kategorisierungsregeln

Diese Regeln ordnen Transaktionen automatisch einer Kategorie zu,
wenn der Empfängername (`payee`) ein Muster enthält.

**Beispiel:** Enthält `payee` den Text `"REWE"` → Kategorie `"supermarkt"` (= Supermarkt)

**Format:** `(Suchmuster, Suchfeld, Ziel-Kategorien-Schlüssel, Priorität)`

- **Suchfeld** ist hier immer `"payee"` (Empfänger) — `"purpose"` (Verwendungszweck) wäre auch möglich
- **Priorität**: höhere Zahl = wird zuerst geprüft. Bei `PAYPAL` (Priorität 1) würde eine spezifischere
  Regel mit höherer Priorität gewinnen, falls vorhanden.

In [ ]:
DEFAULT_RULES = [
    ("REWE",            "payee", "supermarkt",    10),
    ("EDEKA",           "payee", "supermarkt",    10),
    ("LIDL",            "payee", "supermarkt",    10),
    ("ALDI",            "payee", "supermarkt",    10),
    ("PENNY",           "payee", "supermarkt",    10),
    ("NETTO",           "payee", "supermarkt",    10),
    ("KAUFLAND",        "payee", "supermarkt",    10),
    ("NETFLIX",         "payee", "streaming",     10),
    ("SPOTIFY",         "payee", "musik",         10),
    ("AMAZON PRIME",    "payee", "streaming",     10),
    ("DISNEY",          "payee", "streaming",     10),
    ("ARD ZDF",         "payee", "rundfunk",      10),
    ("RUNDFUNK",        "payee", "rundfunk",      10),
    ("TANKSTELLE",      "payee", "tanken",        10),
    ("ARAL",            "payee", "tanken",        10),
    ("SHELL",           "payee", "tanken",        10),
    ("DB BAHN",         "payee", "opnv",          10),
    ("MVGO",            "payee", "opnv",          10),
    ("UBER",            "payee", "taxi",          10),
    ("PAYPAL",          "payee", "sonstiges_a",    1),
]

## Zelle 12 · Datenbank initialisieren (`init_db`)

Diese Funktion wird **einmalig beim App-Start** aufgerufen.

1. `executescript(SCHEMA)` — führt alle `CREATE TABLE IF NOT EXISTS`-Befehle aus.
   Bei einem Neustart: keine Auswirkung (Tabellen existieren bereits).
   Beim allerersten Start: legt alle Tabellen an.

2. Prüfung ob Kategorien-Tabelle leer ist → falls ja: `_seed(conn)` aufrufen.
   Wenn schon Kategorien vorhanden sind (auch eigene), wird kein Seed ausgeführt.

Dieses Muster stellt sicher: Die App ist sofort benutzbar, ohne manuelle Einrichtung.

In [ ]:
def init_db():
    conn = get_db()
    conn.executescript(SCHEMA)
    conn.commit()

    count = conn.execute("SELECT COUNT(*) FROM categories").fetchone()[0]
    if count == 0:
        _seed(conn)

    conn.close()

## Zelle 13 · Standard-Daten einfügen (`_seed`)

Füllt die leere Datenbank mit den Standard-Kategorien und Regeln.

**Warum zwei Durchläufe für Kategorien?**
Weil Untergruppen eine `parent_id` brauchen, die erst bekannt ist, nachdem die Übergruppe
eingefügt und ihre UUID generiert wurde. Deshalb:

1. **Erst Übergruppen** einfügen (alle mit `parent_slug = None`)
   → ihre UUIDs in `slug_to_id` merken
2. **Dann Untergruppen** einfügen — `slug_to_id.get(parent_slug)` liefert die richtige UUID

3. **Dann Regeln** einfügen — `slug_to_id[cat_slug]` gibt die UUID der Zielkategorie

Am Ende: `commit()` — erst jetzt werden die Daten dauerhaft gespeichert.
Falls vorher ein Fehler auftritt, wird nichts geschrieben (Transaktion rollt zurück).

In [ ]:
def _seed(conn):
    ts = now_iso()
    slug_to_id: dict[str, str] = {}

    parents  = [(s, n, p, t, i) for s, n, p, t, i in DEFAULT_CATEGORIES if p is None]
    children = [(s, n, p, t, i) for s, n, p, t, i in DEFAULT_CATEGORIES if p is not None]

    for slug, name, _, cat_type, icon in parents:
        cat_id = uid()
        slug_to_id[slug] = cat_id
        conn.execute(
            "INSERT INTO categories (id, name, parent_id, type, icon, created_at) VALUES (?,?,?,?,?,?)",
            (cat_id, name, None, cat_type, icon, ts),
        )

    for slug, name, parent_slug, cat_type, icon in children:
        cat_id = uid()
        slug_to_id[slug] = cat_id
        conn.execute(
            "INSERT INTO categories (id, name, parent_id, type, icon, created_at) VALUES (?,?,?,?,?,?)",
            (cat_id, name, slug_to_id.get(parent_slug), cat_type, icon, ts),
        )

    for pattern, field, cat_slug, priority in DEFAULT_RULES:
        if cat_slug in slug_to_id:
            conn.execute(
                "INSERT INTO category_rules (id, pattern, field, category_id, priority, created_at) VALUES (?,?,?,?,?,?)",
                (uid(), pattern, field, slug_to_id[cat_slug], priority, ts),
            )

    conn.commit()